# ImMAP-SB vs I2SB

Runs the SAME trained regressor two ways on the same validation slices, with the same noise draws.
Works for both bridges: T1 -> CT1 (the bridge starts at this session's T1) and other CT1 -> CT1
(`x1_source="other_study"`: the bridge starts at another study's CT1, and this session's T1 is
passed to the prox separately as the measurement `y`).

* **I2SB** -- the plain reverse bridge (`sb.i2sb.i2sb_sample`);
* **ImMAP-SB** -- the same bridge, but every endpoint estimate $\hat x = D(z_t, t)$ is replaced by the
  learned data-consistency prox (`sb.immap_sb.immap_sb`)

$$\tilde x = \arg\min_x \tfrac12\|x-\hat x\|^2/\gamma_t + \tfrac12\|M(T1 - A(x))\|^2/\sigma_A^2,
\qquad \gamma_t = c\,\sigma_{\rm eff}(t)^2,$$

solved by Gauss-Newton at $\hat x$ plus CG, before the ordinary posterior update. $A$ is the frozen
learned operator CT1 → T1. Nothing is trained here: this is the plug-and-play comparison.

Because the noise is paired, every difference between the two columns below is the prox.

In [ ]:
import os, sys, math, time
import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
%matplotlib inline

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

# ---- the regressor (a trained i2sb run's SAVED config) and the frozen operator --------------
RUN_CONFIG = "trained_nets/nyumets/I2SB_Unet_NYUMets_CT1_from_all/config.json"
# the other-CT1 bridge (config/NYUMets/i2sb_unet_from_otherct1.json), once trained:
# RUN_CONFIG = "trained_nets/nyumets/I2SB_Unet_NYUMets_CT1_from_otherCT1/config.json"
A_CKPT = "trained_nets/nyumets/forward_ladder_unet_cond_w8_CT1_to_T1/E_unet_w8_l3_x.pt"
A_COND_IDX = [3, 0]       # A's side information IF it is conditioned (ignored for a CT1-only A)
A_DATA = {"image_key": "img_median_mad", "scales": [3.0, 3.0, 3.0, 3.0]}   # what A was trained on
SIGMA_A = None            # None = the val rmse stored in A's checkpoint

# ---- ImMAP-SB prox ---------------------------------------------------------------------------
C = 1.0                   # gamma_t = C * sigma_eff(t)^2   (0 would reproduce I2SB exactly)
T_MAX = 1.0               # prox only for t <= T_MAX
CG_ITERS, CG_TOL, GN_ITERS = 10, 1e-4, 1
USE_MASK = True           # fidelity region M = brain mask (False: whole frame)

# ---- sampling and data -------------------------------------------------------------------------
NFE = None                # None = the run's val_nfe
N_SLICES = 64             # fixed random val subset
SLICE_RANGE = (40, 110)   # original slice indices [lo, hi) to use; overrides the run's config
                          # (older runs have none). None = the run's own setting
BATCH = 8
SEED = 0
ENH_Q = 0.98              # enhancement proxy: top 2% of CT1 - T1 per slice
N_SHOW = 4                # slices in the comparison figure
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", DEVICE)

## Load the regressor, the schedule and A

The regressor and schedule come from the run's own saved config. A's data scaling is not stored in
its checkpoint, so `A_DATA` states it and the cell refuses a mismatch with the run's data.

In [ ]:
import datasets                                    # noqa: F401  (registers loaders)
from datasets.registry import build_loader
from models import build_model
from training.common import load_ckpt
from training.i2sb import _split_batch
from training.forward_op import fixed_val_subset, enh_region
from training.metrics import compute_metrics
from sb.base import build_schedule
from sb.i2sb import i2sb_sample
from sb.immap_sb import ImMAPProx, immap_sb
from sb.learned_dc import _load_E

with open(RUN_CONFIG) as f:
    cfg = yaml.safe_load(f)
if cfg.get("task") != "i2sb":
    raise ValueError(f"{RUN_CONFIG}: task is {cfg.get('task')!r}, not i2sb")
if cfg["i2sb"].get("learned_dc"):
    print("NOTE: this run trained WITH learned DC; here it is used as a plain regressor.")
vcfg = dict(cfg["data"]["val"])
for k, v in A_DATA.items():
    if vcfg.get(k) != v:
        raise ValueError(f"A was trained with {k}={v}, but the run's val data has {k}={vcfg.get(k)}")

net = build_model(cfg).to(DEVICE).eval()
ckpt = cfg["paths"].get("ckpt") or os.path.join(cfg["paths"]["save_dir"], "net.ckpt")
load_ckpt(ckpt, model=net, device=DEVICE)
i2 = cfg["i2sb"]
sched = build_schedule(kind=i2.get("kind", "brownian"), tau=i2.get("tau", 0.19),
                       n_points=i2.get("n_points", 1000), beta_max=i2.get("beta_max", 0.3),
                       device=DEVICE)
if hasattr(net, "assert_schedule_matches"):
    net.assert_schedule_matches(sched)
nfe = NFE or int(i2.get("val_nfe", 20))
samp_kw = dict(nfe=nfe, deterministic=bool(i2.get("deterministic", False)),
               posterior=i2.get("posterior", "ddpm"),
               clip_denoise=bool(i2.get("clip_denoise", False)), verbose=False)
data_range = float(cfg["training"].get("data_range", 1.0))

A, a_rmse = _load_E(A_CKPT, DEVICE)
sigma_A = SIGMA_A if SIGMA_A is not None else a_rmse
if sigma_A is None:
    raise ValueError("A's checkpoint stores no val rmse: set SIGMA_A")
prox = ImMAPProx(sched, A, sigma_A, c=C, t_max=T_MAX, cg_iters=CG_ITERS, gn_iters=GN_ITERS,
                 cg_tol=CG_TOL)
print(f"regressor {type(net).__name__}  |  A cond_channels={A.cond_channels}  sigma_A={sigma_A:.4f}"
      f"  |  nfe={nfe}  c={C}  t_max={T_MAX}  cg={CG_ITERS}  gn={GN_ITERS}")

## Data

The run's own val loader, with `cond_idx` widened to include A's side information if A is
conditioned: the regressor gets exactly the channels it was trained on, A gets its own.

In [ ]:
run_cond = list(vcfg.get("cond_idx") or [])
a_cond = list(A_COND_IDX) if A.cond_channels else []
all_cond = run_cond + [c for c in a_cond if c not in run_cond]
net_sel, a_sel = list(range(len(run_cond))), [all_cond.index(c) for c in a_cond]
vcfg.update(cond_idx=all_cond, batch_size=BATCH)
if SLICE_RANGE is not None:
    vcfg["slice_range"] = list(SLICE_RANGE)
full = build_loader(vcfg, shuffle=False, drop_last=False)
loader = fixed_val_subset(full, N_SLICES, SEED)
x1_source = vcfg.get("x1_source", "contrast")
start = "another study's CT1" if x1_source == "other_study" else "this session's T1"
print(f"{len(loader.dataset)}/{len(full.dataset)} val slices | regressor cond {run_cond} | A cond {a_cond}"
      f" | bridge start: {start}")

## Run both samplers

Per batch, the torch RNG is reset to the same seed before each sampler, so both draw identical
bridge noise. ImMAP-SB's per-step diagnostics (damping $\lambda_t$, data residual before and after
the prox, size of the correction) are kept for the plots further down.

In [ ]:
keep = {"x0": [], "x1": [], "y": [], "m": [], "enh": [], "i2sb": [], "immap": [], "cA": []}
step_stats = []
t_i2sb = t_immap = 0.0

def _sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

with torch.no_grad():
    for bi, batch in enumerate(loader):
        x0, x1, cond, mask, _, _ = _split_batch(batch, DEVICE)
        # the measurement: this session's T1 -- returned as "y" when the bridge starts elsewhere
        y = batch["y"].to(DEVICE) if isinstance(batch, dict) and "y" in batch else x1
        c_net = None if (cond is None or not net_sel) else cond[:, net_sel]
        c_A = None if not a_sel else cond[:, a_sel]
        m = (mask > 0.5).float()
        seed = SEED * 100003 + bi

        torch.manual_seed(seed); t0 = time.time()
        r_i2sb, _, _ = i2sb_sample(net, x1, sched, cond=c_net, **samp_kw)
        _sync(); t_i2sb += time.time() - t0

        torch.manual_seed(seed); t0 = time.time()
        r_immap, _, _, st = immap_sb(net, x1, sched, prox, y=y, cond=c_net, a_cond=c_A,
                                     mask=m if USE_MASK else None, **samp_kw)
        _sync(); t_immap += time.time() - t0
        step_stats += st

        keep["x0"].append(x0.cpu()); keep["x1"].append(x1.cpu()); keep["y"].append(y.cpu())
        keep["m"].append(m.cpu())
        keep["enh"].append(enh_region(x0, y, m, ENH_Q).cpu())
        keep["i2sb"].append(r_i2sb.real.cpu()); keep["immap"].append(r_immap.real.cpu())
        if c_A is not None:
            keep["cA"].append(c_A.cpu())
        print(f"batch {bi + 1}/{len(loader)}")

S = {k: torch.cat(v) for k, v in keep.items() if v}
print(f"\nwall clock: I2SB {t_i2sb:.1f}s   ImMAP-SB {t_immap:.1f}s   ({t_immap / max(t_i2sb, 1e-9):.1f}x)")

## Metrics

Pooled over the brain mask. `t1_res` is the consistency with the measurement, RMS of
$M(T1 - A(\text{sample}))$: ImMAP-SB should bring it toward $\sigma_A$ -- not below it, which would
mean fitting $A$'s own error. `enh` / `rest` split the CT1 error into the enhancement proxy and
everything else.

In [ ]:
def summarize(key):
    x0, y, m, enh, s = S["x0"], S["y"], S["m"], S["enh"], S[key]
    rest = m * (1 - enh)
    e2 = (s - x0) ** 2
    rm = lambda w: math.sqrt(float((e2 * w).sum()) / max(float(w.sum()), 1.0))
    with torch.no_grad():
        cA = S["cA"].to(DEVICE) if "cA" in S else None
        t1 = ((y.to(DEVICE) - A(s.to(DEVICE), cA)) ** 2).cpu()
    mets = compute_metrics(x0 * m, s * m, data_range=data_range, mask=m)
    return {"psnr": float(mets["psnr"]), "ssim": float(mets["ssim"]), "rmse": rm(m),
            "rmse_enh": rm(enh), "rmse_rest": rm(rest),
            "t1_res": math.sqrt(float((t1 * m).sum()) / max(float(m.sum()), 1.0))}

res = {"I2SB": summarize("i2sb"), "ImMAP-SB": summarize("immap")}
cols = ["psnr", "ssim", "rmse", "rmse_enh", "rmse_rest", "t1_res"]
print(f"{'':>10s} " + " ".join(f"{c:>9s}" for c in cols) + f"     (sigma_A = {sigma_A:.4f})")
for name, r in res.items():
    print(f"{name:>10s} " + " ".join(f"{r[c]:9.4f}" for c in cols))

### Per slice

Is the change consistent, or driven by a few slices? Per-slice PSNR of ImMAP-SB minus I2SB: mass to
the right of zero means the prox helps on most slices.

In [ ]:
from visualization.hist import plot_hist

def per_slice_psnr(key):
    out = []
    for i in range(S["x0"].shape[0]):
        m = S["m"][i:i + 1]
        if float(m.sum()) == 0:
            continue
        out.append(float(compute_metrics(S["x0"][i:i + 1] * m, S[key][i:i + 1] * m,
                                         psnr_only=True, data_range=data_range, mask=m)["psnr"]))
    return np.array(out)

d = per_slice_psnr("immap") - per_slice_psnr("i2sb")
print(f"ImMAP-SB - I2SB per-slice PSNR: mean {d.mean():+.3f} dB, median {np.median(d):+.3f} dB, "
      f"better on {100 * (d > 0).mean():.0f}% of {d.size} slices")
plot_hist(d, bins=30, vlines={"no change": 0.0}, xlabel="PSNR(ImMAP-SB) - PSNR(I2SB)  [dB]",
          title="per-slice PSNR difference", show=True)

## Slices

Columns: T1 (the measurement y), the bridge start x1 when it is another study's CT1, CT1 (the
target), the two samples, their errors against CT1 on one
fixed diverging window, and what the prox changed (ImMAP-SB − I2SB). The last column is where to
look: it should be concentrated in anatomy, and it cannot put enhancement in -- the enhancement is
in A's null space, so it is still the regressor's job.

In [ ]:
from visualization.image import subplot_images

idx = list(range(min(N_SHOW, S["x0"].shape[0])))
two_starts = not torch.equal(S["x1"], S["y"])     # bridge starts somewhere other than T1
n_img = 5 if two_starts else 4
err_ref = (S["i2sb"][idx] - S["x0"][idx])[S["m"][idx] > 0.5]
v = float(torch.quantile(err_ref.abs().float(), 0.99)) if err_ref.numel() else 1.0
rows, labels = [], []
for i in idx:
    x0, x1, yy, a, b, m = (S[k][i, 0] for k in ("x0", "x1", "y", "i2sb", "immap", "m"))
    lead = [yy, x1] if two_starts else [yy]
    rows.append(lead + [x0, a, b, a - x0, b - x0, b - a])
    labels.append(f"slice {i}")
fig, _ = subplot_images(
    rows, row_labels=labels,
    col_titles=(["T1 (y)"] + (["x1: other CT1"] if two_starts else [])
                + ["CT1 (target)", "I2SB", "ImMAP-SB", "I2SB − CT1", "ImMAP-SB − CT1",
                   "ImMAP-SB − I2SB"]),
    cmap=["gray"] * n_img + ["RdBu_r"] * 3,
    vmin=[None] * n_img + [-v] * 3, vmax=[None] * n_img + [v] * 3,
    window_from=[S["x0"][idx]], p=(1, 99), mask=S["m"][idx], apply_mask=True,
    magnitude=False, panel_size=(2.4, 2.6), show=False)
plt.show()

## What the prox did along the bridge

Per visited step (pooled over batches): the damping $\lambda_t = \sigma_A^2/\gamma_t$, the data
residual $\|M(T1 - A(\cdot))\|$ before and after the prox, and the RMS size of the correction
$\|\tilde x - \hat x\|$. Near $t = 1$, $\lambda_t$ is tiny and the prox is close to a pure fit of
$A(x) = T1$; near $t = 0$ it is large and the prox barely moves $\hat x$.

In [ ]:
act = [s for s in step_stats if s.get("active")]
if not act:
    print("the prox was never active (C = 0 or T_MAX too small)")
else:
    n = max(1, sched.std_fwd.shape[0] - 1)
    steps = sorted({s["step"] for s in act})
    agg = lambda key: [np.mean([s[key] for s in act if s["step"] == k]) for k in steps]
    t = [k / n for k in steps]
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
    ax[0].semilogy(t, agg("lam"), "o-"); ax[0].set_title(r"damping $\lambda_t$")
    ax[1].plot(t, agg("res_before"), "o-", label="before prox (at $\\hat x$)")
    ax[1].plot(t, agg("res_after"), "s-", label="after prox")
    ax[1].axhline(sigma_A, ls="--", c="k", label=r"$\sigma_A$")
    ax[1].set_title(r"data residual $\|M(T1 - A)\|$"); ax[1].legend(fontsize=8)
    ax[2].plot(t, agg("delta_rms"), "o-"); ax[2].set_title(r"correction $\|\tilde x - \hat x\|$")
    for a in ax:
        a.set_xlabel("bridge position t  (0 = CT1, 1 = T1)"); a.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

## Reading it

* **ImMAP-SB better on PSNR/`rmse_rest`, `enh` unchanged**: the expected outcome -- the prox anchors
  anatomy to T1 and leaves enhancement to the regressor.
* **`t1_res` well below $\sigma_A$, or samples drifting toward T1**: the prox is too strong at large
  $t$. Lower `C`, or set `T_MAX` below 1.
* **No change at all**: `C` too small, or the regressor is already consistent with T1 (likely for a
  regressor that sees FLAIR/T1/T2). A T1-only regressor is where the prox has the most room.
* **Other-CT1 bridge**: this is where the prox should matter -- the start carries the WRONG
  anatomy, and the prox is strongest (smallest $\lambda_t$) at exactly those first steps. Watch the
  enhancement too: the prox cannot correct enhancement inherited from the other study (it is in
  A's null space), so check lesions that differ between the two studies.
* For a sweep over `C`, `scripts/eval_immap_sb.py` does several values in one run.